In [ ]:
import numpy as np, os, time
import xarray as xa
from argparse import Namespace
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')
opts.defaults( opts.Curve(width=1000, height=600, line_width=1) )
import tmodel

signal_index = 2
args: Namespace = tmodel.load_args(signal_index, 0)
att_path = tmodel.attribution_path( args, signal_index )
att_ds: xa.Dataset = xa.open_dataset( att_path )

data=tmodel.get_demo_data()
T: np.ndarray = data['times'][signal_index]

In [ ]:
curve_dict = {}
for vname, avar in att_ds.data_vars.items():
	iFT = int(vname[2:])
	Att0 = avar.values[0]
	nF = avar.shape[0]
	T0 = T[:Att0.size]
	C0 = hv.Curve((T0, Att0), 'Time', 'Amplitude', label='Full Result').opts(color="green", alpha=0.6)
	for iF in range( 1, nF ):
		Att = avar.values[iF]
		curve_dict[(iFT,iF)] = hv.Curve((T0, Att), 'Time', 'Amplitude', label=f'Feature {iF} Masked').opts(color="red", alpha=0.6) * C0

In [ ]:
kdims = [ hv.Dimension(('ftype', 'Feature Type'), default=0), hv.Dimension(('feature', 'Feature'), default=0) ]
holomap = hv.HoloMap(curve_dict, kdims=kdims)
holomap.opts(opts.Curve(width=1500, title="Masked Feature Results", ylim=(0.92, 1.02), yticks=np.arange(0.92, 1.02, 0.02) ))

In [ ]:
iFT = 0
grid = hv.GridSpace( { (ir,ic): curve_dict[(iFT,ir*4+ic)] for ir in range(2) for ic in range(4) } )
grid.opts(opts.Curve(width=1500, title="Masked Feature Results", ylim=(0.92, 1.02), yticks=np.arange(0.92, 1.02, 0.02) ))